In [41]:
import torch, random
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("backfeed.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))

s_i = {s:i+1 for i, s in enumerate(chars)}
s_i["."] = 0

i_s = {i:s for s, i in s_i.items()}

# building dataset

block_size = 3

def build_dataset(words):
    
    X, Y = [], []
    for w in words:
        # print(w)
        context = [0] * block_size
        
        for ch in w + ".":
            ix = s_i[ch]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]
            
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


In [48]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)

w1 = torch.randn((30, 200))
b1 = torch.randn(200)

w2 = torch.randn((200, 27))
b2 = torch.randn(27)

parameters = [C, w1, b1, w2, b2]

for p in parameters:
    p.requires_grad = True

""" lre = torch.linspace(-3, 0, 1000)
lrs = 10 ** lre

lri = [] """
lossi = []
stepi = []

In [50]:
# training loop
max_steps = 50000
for i in range(max_steps):
    # mini batch
    ix = torch.randint(0, Xtr.shape[0], (32,))
    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 30) @ w1 + b1)

    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    # backward pass
    for p in parameters:
        p.grad = None
        
    loss.backward()
    
    # update
    # lr = lrs[i]
    lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad
      
    # track stats  
    # lri.append(lre[i])
    if i % 10000 == 0:
        print(f"{i:7d}/{max_steps:7d}: {loss.item():4f}")
    stepi.append(i)
    lossi.append(loss.log10().item())

      0/  50000: 2.274688
  10000/  50000: 1.837171
  20000/  50000: 1.566156
  30000/  50000: 2.225383
  40000/  50000: 2.101950


In [51]:
loss.item()

1.9584134817123413

In [53]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 30) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, Ydev)
print(loss.item())

2.174720048904419


In [73]:
for p in parameters:
    p.data *= 0.5

In [74]:
g2 = torch.Generator().manual_seed(2147483647 + 10)
names = []

for _ in range(20):
    out = []
    context = [0] * block_size
    
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(-1, 30) @ w1 + b1)
        logits = h @ w2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g2).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(ix)
        
    
    print("".join(i_s[i] for i in out))
        

carmah
amella
kazimis
jah
casrafelie
azhith
ames
aurian
aure
ania
cariim
arie
gavi
madiia
qui
tan
lin
ania
brios
tafediarisi


In [40]:
g2 = torch.Generator().manual_seed(2147483647 + 10)
names = []

with open("backfeed.txt", "w") as file:
    for _ in range(32033):
        out = []
        context = [0] * block_size
        
        while True:
            emb = C[torch.tensor([context])]
            h = torch.tanh(emb.view(-1, 30) @ w1 + b1)
            logits = h @ w2 + b2
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g2).item()
            context = context[1:] + [ix]
            if ix == 0:
                break
            out.append(ix)
            
        
        names.append("".join(i_s[i] for i in out))
        file.write(f"{"".join(i_s[i] for i in out)}\n")
        